# COMPASS-XAI: End-to-End Market Expansion Command Center

This notebook serves as the primary orchestrator for the **COMPASS-XAI** platform. It executes the entire analytical pipeline—from data ingestion and cleaning to demand forecasting and strategic explainability.

## 10-Step Framework Mapping (DAP391m)
| Step | Phase | Core Component |
| :--- | :--- | :--- |
| 1-3 | Understanding | Business Context & Data Requirements |
| 4-5 | Ingestion | `RawDataLoader` & IBGE External Joins |
| 6 | Preparation | `DataService` (Cleaning) & `FeatureEngineeringService` |
| 7-8 | Modeling | `ModelingService` & `ModelEvaluator` (Walk-forward CV) |
| 9 | Deployment | `ScoringService` (Entropy Optimization) |
| 10 | Feedback | `XAIService` (Gemini-aligned Narratives) |

### 1. Initialization & Configuration
We load the global configuration using Pydantic Settings to ensure environment portability and strict type validation.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path for local package imports
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

from src.olist_pipeline.core.config import get_config
from src.olist_pipeline.core.logging_setup import setup_logger
from src.olist_pipeline.data.loader import RawDataLoader
from src.olist_pipeline.data.cleaning import DataService
from src.olist_pipeline.features.engineering import FeatureEngineeringService
from src.olist_pipeline.models.training import ModelingService
from src.olist_pipeline.analysis.scoring import ScoringService
from src.olist_pipeline.analysis.xai import XAIService
from src.olist_pipeline.analysis.providers.gemini import GeminiProvider

config = get_config()
paths = config.paths
logger = setup_logger("notebook_orchestrator")

print(f"Successfully initialized COMPASS-XAI at: {PROJECT_ROOT}")

### 2. Data Ingestion & Cleaning
We use the `DataService` to filter for delivered orders, impute product dimensions, and validate the schema of the 100k+ Olist transactions.

In [ ]:
logger.info("Starting Step 4/6: Ingestion and Cleaning")

# 1. Automated download from Kaggle if data is missing
loader = RawDataLoader(paths.data.raw_olist)
loader.download_if_missing()

# 2. Domain-specific cleaning
data_svc = DataService(paths.data.raw_olist, paths.data.processed_olist)
data_svc.run_cleaning_pipeline()

logger.info("Cleaning complete. Processed files saved to data/processed/olist/")

### 3. Feature Engineering (State-Week Panel)
We aggregate transactions into weekly state-level blocks and join them with external IBGE population and GDP per-capita data.

In [ ]:
logger.info("Starting Step 6: Feature Engineering Service")

feat_svc = FeatureEngineeringService(
    paths.data.processed_olist,
    paths.data.processed_olist / "features_weekly.csv",
    paths.data.processed_olist / "prediction_data.csv"
)
feat_svc.run_feature_pipeline(paths.data.external_pop, paths.data.external_gdp)

df_features = pd.read_csv(paths.data.processed_olist / "features_weekly.csv")
print(f"\nFeature Matrix created with {df_features.shape[0]} rows and {df_features.shape[1]} columns.")
display(df_features.head())

### 4. Machine Learning: Forecasting Next-Week Demand
We benchmark 8 models using **walk-forward cross-validation**. The model with the best RMSE is promoted as the champion explainer.

In [ ]:
logger.info("Starting Step 7/8: Modeling and Benchmarking")

model_svc = ModelingService(
    config.training.model_dump(), 
    paths.reports.figures_dir.parent
)
model_svc.run_training_pipeline(paths.data.processed_olist / "features_weekly.csv")

leaderboard = pd.read_csv(paths.reports.leaderboard)
print("\nModel Performance Leaderboard:")
display(leaderboard.sort_values("RMSE"))

### 5. Strategic Scoring: Expansion Priority Score (EPS)
The `ScoringService` calculates normalized components (Demand, Growth, Gap, Momentum) and finds optimal weights by maximizing **Shannon Entropy**.

In [ ]:
logger.info("Starting Step 9: Scoring Engine (SLSQP Optimization)")

score_svc = ScoringService(config.inference.model_dump(), paths.outputs.eps_dir)
score_svc.run_scoring_pipeline(
    paths.data.processed_olist / "features_weekly.csv",
    paths.data.processed_olist / "prediction_data.csv"
)

df_eps = pd.read_csv(paths.outputs.eps_results)
print("\nTop 10 High-Priority Expansion States:")
display(df_eps.sort_values("EPS_rank").head(10)[["customer_state", "EPS_score", "EPS_rank", "tier", "dominant_component"]])

### 6. Explainable AI: COMPASS Narratives
We synthesize the final rankings with SHAP feature attributions and use Google Gemini to generate human-readable strategic summaries.

In [ ]:
logger.info("Starting Step 10: XAI Narrative Synthesis")

llm_key = os.getenv("GEMINI_API_KEY", "")
llm = GeminiProvider(api_key=llm_key)

xai_svc = XAIService(paths.outputs.eps_dir, llm_provider=llm)
xai_svc.run_xai_pipeline(paths.outputs.eps_results, paths.outputs.w_star)

print("\nStrategic reports and SHAP profiles generated in outputs/eps/.")

### 7. Global Insights Visualization
A final view of the geographic distribution of expansion opportunities across Brazil.

In [ ]:
plt.figure(figsize=(14, 7))
sns.set_style("whitegrid")

plot_df = df_eps.sort_values("EPS_score", ascending=False)
ax = sns.barplot(data=plot_df, x="customer_state", y="EPS_score", hue="tier", palette="magma")

plt.title("COMPASS-XAI: Expansion Priority Score (EPS) Rankings", fontsize=15, pad=20)
plt.xlabel("Brazilian State (UF)", fontsize=12)
plt.ylabel("EPS Score (0-100)", fontsize=12)
plt.legend(title="Strategy Tier")
plt.show()